In [2]:
import sqlite3
import os
from tqdm import tqdm
import pandas as pd
import pyBigWig as bw
import subprocess

In [2]:
# Path to the folder containing .db files
folder_path = "/data1/normantm/eli/scratch/chr_motifs/"

# Dictionary to store counts
db_counts = {}

# Loop through all .db files and count entries
for file_name in tqdm(os.listdir(folder_path)):
    if file_name.endswith(".db"):
        db_path = os.path.join(folder_path, file_name)
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Get table name dynamically
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        table_name = cursor.fetchone()[0]  # Assumes one table per DB

        # Count entries in the table
        cursor.execute(f"SELECT COUNT(*) FROM {table_name};")
        count = cursor.fetchone()[0]

        # Store in dictionary
        db_counts[file_name] = count

        conn.close()

# Print results
for db, count in db_counts.items():
    print(f"{db}: {count} entries", flush = True)

  0%|          | 0/24 [00:00<?, ?it/s]

100%|██████████| 24/24 [21:19<00:00, 53.30s/it]  

chr22_motifs.db: 199447402 entries
chr13_motifs.db: 620603838 entries
chr18_motifs.db: 479639360 entries
chrX_motifs.db: 937433740 entries
chr16_motifs.db: 449368035 entries
motifs.db: 17358421683 entries
chr21_motifs.db: 235739821 entries
chr1_motifs.db: 1356465349 entries
chr6_motifs.db: 1048898131 entries
chr10_motifs.db: 783227722 entries
chr11_motifs.db: 785421628 entries
chr9_motifs.db: 718716553 entries
chr15_motifs.db: 484782007 entries
chr20_motifs.db: 352154225 entries
chr7_motifs.db: 953029286 entries
chr2_motifs.db: 1465401861 entries
chr4_motifs.db: 1209211582 entries
chr17_motifs.db: 444304548 entries
chr14_motifs.db: 539952619 entries
chr8_motifs.db: 878495141 entries
chr12_motifs.db: 795499537 entries
chr19_motifs.db: 295609473 entries
chr5_motifs.db: 1112018219 entries
chr3_motifs.db: 1213001606 entries


In [3]:
motif_counts = db_counts.pop("motifs.db")
total_counts = sum(db_counts.values())
motif_counts == total_counts

True

In [20]:
for file in os.scandir("/data1/normantm/eli/scratch/chr_motifs"):
    if file.name.endswith(".db"):
        conn = sqlite3.connect(file.path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        name = cursor.fetchone()[0]
        print(name)
        conn.close()

motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs
motifs


In [4]:
targets = []
for file in os.scandir("/data1/normantm/eli/scratch/sc_bws"):
    files = os.listdir(file.path)
    if not any([f.endswith(".bw") for f in files]):
        targets.append(file.name)

In [6]:
targets = [t for t in targets if t not in ['MYRF.bam', 'PRDM16']]
len(targets)

94

In [16]:
cmd = "srun --cpus-per-task=32 --mem-per-cpu=24G --time=1-00:00:00 python /data1/normantm/eli/202412_expt/sc_bws.py --target "
for i in range(0, len(targets), 10):
    print(cmd + ' '.join(targets[i:i+10]))

srun --cpus-per-task=32 --mem-per-cpu=24G --time=1-00:00:00 python /data1/normantm/eli/202412_expt/sc_bws.py --target HOXB6 RUNX3 BCL11A ZFX OLIG2 TLX1 RXRG FOXQ1 NR2E1 GLI1
srun --cpus-per-task=32 --mem-per-cpu=24G --time=1-00:00:00 python /data1/normantm/eli/202412_expt/sc_bws.py --target ONECUT1 VSX1 DLX3 NR4A3 ATOH7 ONECUT3 ZNF813 TEAD3 HIF1A EGR2
srun --cpus-per-task=32 --mem-per-cpu=24G --time=1-00:00:00 python /data1/normantm/eli/202412_expt/sc_bws.py --target ONECUT2 DLX2 HMX3 TGIF1 POU4F1 CIITA GATA2 FOXC1 FOS NKX2-5
srun --cpus-per-task=32 --mem-per-cpu=24G --time=1-00:00:00 python /data1/normantm/eli/202412_expt/sc_bws.py --target FOSB NEUROD1 LBX1 TOX OSR1 SIX3 KLF4 FOXL2 PRDM1 CRX
srun --cpus-per-task=32 --mem-per-cpu=24G --time=1-00:00:00 python /data1/normantm/eli/202412_expt/sc_bws.py --target ARID1A LHX4 OSR2 KLF16 SOX9 FOXN4 PAX6 POU4F2 SOX13 EPAS1
srun --cpus-per-task=32 --mem-per-cpu=24G --time=1-00:00:00 python /data1/normantm/eli/202412_expt/sc_bws.py --target GLI

In [9]:
scores = pd.read_csv("~/ALX_test.txt", sep = '\t', header = None)
scores.columns = ['region', 'name', 'size', 'covered', 'mean0', 'mean']
scores['mean'].mean()

0.007713670252786102

In [10]:
scores = pd.read_csv("~/ALX_test_NTC.txt", sep = '\t', header = None)
scores.columns = ['region', 'name', 'size', 'covered', 'mean0', 'mean']
scores['mean'].mean()

0.007273981844578139

In [5]:
coverage = bw.open("/data1/normantm/eli/scratch/sc_bws/FOXC2/AATCCATCAGCACCAT-3_.bw")
bed = pd.read_csv("/data1/normantm/eli/scratch/motif_beds/FOXC2.bed", sep = '\t', header = None)
bed.columns = ['chr', 'start', 'end', 'name']

In [ ]:
signal = 0
for i, row in tqdm(bed.iterrows(), total = len(bed)):
    chr = row['chr']
    start = row['start']
    end = row['end']
    signal += coverage.stats(chr, start, end)[0]

In [3]:
out = subprocess.run("bigWigAverageOverBed /data1/normantm/eli/scratch/sc_bws/FOXC2/AATCCATCAGCACCAT-3_.bw /data1/normantm/eli/scratch/motif_beds/FOXC2.bed /dev/stdout", shell = True, capture_output = True)

In [7]:
import numpy as np
mean = np.mean([float(result.split('\t')[-1]) for result in out.stdout.decode('utf-8').split('\n')[1:-1]])
mean

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

0.007320390664587853

In [ ]:
plt.figure(figsize = (6,3))
df = pd.read_csv("/data1/normantm/eli/FOXF2.csv", index_col=0)
ntc_mean, ntc_sd = df.query("guide_target == 'NTC'").FOXF2.mean(), df.query("guide_target == 'NTC'").FOXF2.std()
df['z'] = (df.FOXF2 - ntc_mean) / ntc_sd
sorted_order = df.groupby('guide_target').z.mean().sort_values().index
sns.barplot(df, x = 'guide_target', y = 'z', order = sorted_order[::-1])
plt.xticks(rotation = 90)
plt.title('Motif: FOXF2')
plt.ylabel("z-normalized enrichment score")
plt.show()

In [ ]:
plt.figure(figsize = (6,3))

df = pd.read_csv("/data1/normantm/eli/DLX3.csv", index_col=0)
ntc_mean, ntc_sd = df.query("guide_target == 'NTC'").Dlx3.mean(), df.query("guide_target == 'NTC'").Dlx3.std()
df['z'] = (df.Dlx3 - ntc_mean) / ntc_sd
sorted_order = df.groupby('guide_target').z.mean().sort_values().index
sns.barplot(df, x = 'guide_target', y = 'z', order = sorted_order[::-1])
plt.xticks(rotation = 90)
plt.title('Motif: DLX3')
plt.ylabel("z-normalized enrichment score")
plt.show()

In [5]:
ont_motifs = pd.read_csv("/data1/normantm/eli/seq2gex/data/get_meanpop_log2.csv", index_col=0).drop("NTC", axis = 0).index
motif_beds = [f.name.split(".")[0] for f in os.scandir("/data1/normantm/eli/scratch/motif_beds") if f.name.endswith(".bed") and f.name.split(".")[0].casefold() not in [m.casefold() for m in ont_motifs]]
motifs = sorted(motif_beds, key = lambda x: x.split("/")[-1].split(".")[0].casefold())
motifs

['Ahr::Arnt',
 'Alx1',
 'ALX3',
 'Ar',
 'ARGFX',
 'Arid3b',
 'Arid5a',
 'Arnt',
 'ARNT2',
 'ARNT::HIF1A',
 'Arntl',
 'Arx',
 'ASCL1',
 'Ascl2',
 'Atf1',
 'ATF2',
 'ATF3',
 'Atf3',
 'ATF4',
 'ATF6',
 'ATF7',
 'Atoh1',
 'BACH1',
 'Bach1::Mafk',
 'BACH2',
 'BARHL1',
 'BARHL2',
 'BARX1',
 'BARX2',
 'BATF',
 'BATF3',
 'BATF::JUN',
 'BCL6',
 'BCL6B',
 'Bhlha15',
 'BHLHA15',
 'BHLHE22',
 'BHLHE23',
 'BHLHE40',
 'BHLHE41',
 'BSX',
 'CDX1',
 'CDX2',
 'CDX4',
 'CEBPD',
 'CEBPE',
 'CEBPG',
 'CLOCK',
 'CREB1',
 'CREB3',
 'CREB3L1',
 'Creb3l2',
 'CREB3L4',
 'Creb5',
 'CREM',
 'CTCF',
 'CTCFL',
 'CUX1',
 'CUX2',
 'DBP',
 'Ddit3::Cebpa',
 'DLX1',
 'Dlx4',
 'Dlx5',
 'DLX6',
 'Dmbx1',
 'Dmrt1',
 'DMRT3',
 'DMRTA1',
 'DMRTA2',
 'DMRTC2',
 'DPRX',
 'DRGX',
 'Dux',
 'DUX4',
 'DUXA',
 'E2F1',
 'E2F2',
 'E2F3',
 'E2F4',
 'E2F6',
 'E2F7',
 'E2F8',
 'EBF1',
 'Ebf2',
 'EBF3',
 'Ebf4',
 'EGR1',
 'EGR4',
 'EHF',
 'ELF2',
 'ELF3',
 'ELF4',
 'Elf5',
 'ELK1',
 'ELK1::HOXA1',
 'ELK1::HOXB13',
 'ELK1::SREBF2',
 'ELK3

In [6]:
len(motifs)

728

In [7]:
len(set(m.casefold() for m in motifs))

718

In [8]:
motifs.remove("Atf3")

In [10]:
# cmd = f"\"python /data1/normantm/eli/202412_expt/sc_motif_scores.py --target all -p 48 -o /data1/normantm/eli/202412_expt/ont_sc_motif_scores_{i}.csv --motif {m}\""
for i in range(0, len(motifs), 7):
    m = ' '.join(motifs[i:i+7])
    cmd = f"\"python /data1/normantm/eli/202412_expt/sc_motif_scores.py --target all -p 48 -o /data1/normantm/eli/202412_expt/ont_sc_motif_scores_{14 + i//7}.csv --motif {m}\""
    # print(f"{cmd} {' '.join(motifs[i:i+7])}\"")
    print(cmd)

"python /data1/normantm/eli/202412_expt/sc_motif_scores.py --target all -p 48 -o /data1/normantm/eli/202412_expt/ont_sc_motif_scores_14.csv --motif Ahr::Arnt Alx1 ALX3 Ar ARGFX Arid3b Arid5a"
"python /data1/normantm/eli/202412_expt/sc_motif_scores.py --target all -p 48 -o /data1/normantm/eli/202412_expt/ont_sc_motif_scores_15.csv --motif Arnt ARNT2 ARNT::HIF1A Arntl Arx ASCL1 Ascl2"
"python /data1/normantm/eli/202412_expt/sc_motif_scores.py --target all -p 48 -o /data1/normantm/eli/202412_expt/ont_sc_motif_scores_16.csv --motif Atf1 ATF2 ATF3 ATF4 ATF6 ATF7 Atoh1"
"python /data1/normantm/eli/202412_expt/sc_motif_scores.py --target all -p 48 -o /data1/normantm/eli/202412_expt/ont_sc_motif_scores_17.csv --motif BACH1 Bach1::Mafk BACH2 BARHL1 BARHL2 BARX1 BARX2"
"python /data1/normantm/eli/202412_expt/sc_motif_scores.py --target all -p 48 -o /data1/normantm/eli/202412_expt/ont_sc_motif_scores_18.csv --motif BATF BATF3 BATF::JUN BCL6 BCL6B Bhlha15 BHLHA15"
"python /data1/normantm/eli/20241

In [11]:
len(range(0, len(motifs), 7))

104